# nb_gold_obras_seont_os
Cria a tabela `gold_obras_seont_os` para alimentar o dashboard Power BI do SEONT.

**Fonte:** `gold_pdr_acompanhamentos_os` + `silver_acto_gesta_obras_santos_solicitacoes.parquet`

**Destino:** tabela Delta `gold_obras_seont_os`

**Novas colunas em relação à gold base:**
- `analista_z1`, `analista_z2`, `analista_z3` — analistas responsáveis por zona
- `executor_responsavel` — réplica da fórmula Aux_Analista2 do Excel
- `flag_seont` — 1 se a etapa atual é de responsabilidade do SEONT
- `flag_etapa_aprov` — 1 se a OS chegou em uma etapa de conclusão/aprovação

## 1. Carregamento dos dados

In [1]:
import pandas as pd
from pyspark.sql import functions as F

# ============================================================
# Tabela gold base (já processada pelo nb_gold_acto_gestao_obras)
# ============================================================
print("📥 Carregando gold_pdr_acompanhamentos_os...")
df = spark.table("gold_pdr_acompanhamentos_os").toPandas()
print(f"✅ Gold base: {len(df):,} registros | {len(df.columns)} colunas")
print(f"   Colunas: {df.columns.tolist()}")
print()

# ============================================================
# Silver solicitacoes — necessário para extrair analistas por zona
# ============================================================
SILVER_SOLICITACOES = "/lakehouse/default/Files/acto_obras/silver/silver_acto_gesta_obras_santos_solicitacoes.parquet"

print("📥 Carregando silver solicitacoes...")
df_solicitacoes = pd.read_parquet(SILVER_SOLICITACOES)
print(f"✅ Silver solicitacoes: {len(df_solicitacoes):,} registros | {len(df_solicitacoes.columns)} colunas")

StatementMeta(, d56499b8-ffb4-4190-8ea8-2cacac44b8d6, 3, Finished, Available, Finished, False)

📥 Carregando gold_pdr_acompanhamentos_os...
✅ Gold base: 10,926 registros | 18 colunas
   Colunas: ['n_da_solicitacao', 'servico', 'status', 'data_criacao', 'data_finalizacao', 'solicitante', 'Executor', 'Etapa', 'titulo_profissional', 'etapa_atual', 'executor_atual', 'aux_setor_responsavel', 'data_etapa_inicio', 'data_etapa_fim', 'tempo_execucao', 'dias_na_etapa', 'zona', 'bairro_consolidado']

📥 Carregando silver solicitacoes...
✅ Silver solicitacoes: 10,926 registros | 375 colunas


## 2. Extração dos analistas por zona (bfill horizontal)

In [2]:
import unicodedata

def normalizar(texto):
    return unicodedata.normalize("NFD", str(texto)).encode("ascii", "ignore").decode("ascii").upper().strip()

def bfill_cols(df, cols, col_destino):
    if not cols:
        df[col_destino] = None
        return df
    df[col_destino] = df[cols].bfill(axis=1).iloc[:, 0]
    return df


# ============================================================
# Localizar colunas do silver (padrao real: "Nome do Campo|ID")
# ============================================================
print("Mapeando colunas do silver...")

cols_id = [c for c in df_solicitacoes.columns if normalizar(c).startswith(normalizar("No Solicitacao")) or normalizar(c).startswith("N") and "SOLICIT" in normalizar(c)]

# busca mais robusta pelo ID
cols_id = [
    c for c in df_solicitacoes.columns
    if normalizar(c).replace(" ", "").startswith("NOSOLICITACAO")
    or normalizar(c).replace(" ", "").startswith("NSOLICITACAO")
]
print("Colunas ID encontradas: " + str(len(cols_id)))
print(str(cols_id))

PADRAO_ANALISTA = normalizar("Esta solicitacao devera ser analisada por:")
cols_analista = [
    c for c in df_solicitacoes.columns
    if normalizar(c).replace(" ", "").startswith(normalizar("Estasolicitacaodevera"))
    or "ANALISADA POR" in normalizar(c)
]
print("Colunas analista encontradas: " + str(len(cols_analista)))
print(str(cols_analista))


# ============================================================
# Consolidar via bfill
# ============================================================
print("Consolidando colunas via bfill...")

df_analistas = df_solicitacoes[cols_id + cols_analista].copy()

# ID: bfill + extrai digitos
df_analistas = bfill_cols(df_analistas, cols_id, "_id_raw")
df_analistas = df_analistas.drop(columns=cols_id)
df_analistas["n_da_solicitacao"] = (
    df_analistas["_id_raw"]
    .astype(str)
    .str.extract(r"(\d+)", expand=False)
)
df_analistas["n_da_solicitacao"] = pd.to_numeric(
    df_analistas["n_da_solicitacao"], errors="coerce"
).astype("Int64")
df_analistas = df_analistas.drop(columns=["_id_raw"])

# Analista: bfill de todas as variantes
df_analistas = bfill_cols(df_analistas, cols_analista, "analista_responsavel")
if cols_analista:
    df_analistas = df_analistas.drop(columns=cols_analista)

# Limpeza
nulos_id = df_analistas["n_da_solicitacao"].isna().sum()
if nulos_id > 0:
    print("Linhas sem ID removidas: " + str(nulos_id))
    df_analistas = df_analistas.dropna(subset=["n_da_solicitacao"])

df_analistas = df_analistas.drop_duplicates(subset=["n_da_solicitacao"])

preench = df_analistas["analista_responsavel"].notna().sum()
total = len(df_analistas)
pct = round(preench / total * 100, 1) if total > 0 else 0
print("analista_responsavel: " + str(preench) + "/" + str(total) + " preenchidos (" + str(pct) + "%)")
print("Analistas extraidos: " + str(total) + " registros | Colunas: " + str(df_analistas.columns.tolist()))

StatementMeta(, d56499b8-ffb4-4190-8ea8-2cacac44b8d6, 4, Finished, Available, Finished, False)

Mapeando colunas do silver...
Colunas ID encontradas: 26
['Nº Solicitação|1', 'Nº Solicitação|2', 'Nº Solicitação|5', 'Nº Solicitação|8', 'Nº Solicitação|11', 'Nº Solicitação|14', 'Nº Solicitação|17', 'Nº Solicitação|20', 'Nº Solicitação|26', 'Nº Solicitação|30', 'Nº Solicitação|34', 'Nº Solicitação|41', 'Nº Solicitação|47', 'Nº Solicitação|51', 'Nº Solicitação|62', 'Nº Solicitação|66', 'Nº Solicitação|72', 'Nº Solicitação|83', 'Nº Solicitação|89', 'Nº Solicitação|109', 'Nº Solicitação|118', 'Nº Solicitação|142', 'Nº Solicitação|155', 'Nº Solicitação|162', 'Nº Solicitação|193', 'Nº Solicitação|200']
Colunas analista encontradas: 11
['Esta solicitação deverá ser analisada por:|385', 'Esta solicitação deverá ser analisada por:|411', 'Esta solicitação deverá ser analisada por:|432', 'Esta solicitação deverá ser analisada por:|453', 'Esta solicitação deverá ser analisada por:|428', 'Esta solicitação deverá ser analisada por:|454', 'Esta solicitação deverá ser analisada por:|477', 'Esta sol

## 3. Merge: gold base + analistas

In [3]:
print("Realizando merge gold + analistas...")

# Evita conflito _x/_y caso o notebook seja re-executado
if "analista_responsavel" in df.columns:
    df = df.drop(columns=["analista_responsavel"])

df["n_da_solicitacao"] = pd.to_numeric(df["n_da_solicitacao"], errors="coerce").astype("Int64")

df = df.merge(df_analistas, on="n_da_solicitacao", how="left")

preenchidos = df["analista_responsavel"].notna().sum() if "analista_responsavel" in df.columns else 0
pct = round(preenchidos / len(df) * 100, 1)
print("analista_responsavel: " + str(preenchidos) + " preenchidos (" + str(pct) + "%)")
print("Merge concluido: " + str(len(df)) + " registros | " + str(len(df.columns)) + " colunas")

StatementMeta(, d56499b8-ffb4-4190-8ea8-2cacac44b8d6, 5, Finished, Available, Finished, False)

Realizando merge gold + analistas...
analista_responsavel: 377 preenchidos (3.5%)
Merge concluido: 10926 registros | 19 colunas


In [4]:
## 4. Cálculo das colunas derivadas

StatementMeta(, d56499b8-ffb4-4190-8ea8-2cacac44b8d6, 6, Finished, Available, Finished, False)

In [5]:
# ============================================================
# 4a. flag_seont
# Identifica se a etapa atual é de responsabilidade do SEONT
# Réplica da lógica do auxiliar PMS_AuxiliarPDR (AuxSetorResponsável)
# ============================================================
print("🏷️ Calculando flag_seont...")

SETORES_SEONT = {"SEONT", "SEONT-Chefia", "SEONT-Chefia (D.O)", "SEONT CHEFIA"}

df["flag_seont"] = (
    df["aux_setor_responsavel"]
    .astype(str)
    .str.strip()
    .isin(SETORES_SEONT)
    .astype(int)
)

print(f"   flag_seont=1: {df['flag_seont'].sum():,} registros")
print(f"   flag_seont=0: {(df['flag_seont'] == 0).sum():,} registros")
print()

StatementMeta(, d56499b8-ffb4-4190-8ea8-2cacac44b8d6, 7, Finished, Available, Finished, False)

🏷️ Calculando flag_seont...
   flag_seont=1: 196 registros
   flag_seont=0: 10,730 registros



In [6]:
# ============================================================
# 4b. flag_etapa_aprov
# Réplica da fórmula PROCV da coluna AJ do Excel (Aux etapa aprov)
# Comparação sem acento para evitar divergências de encoding
# ============================================================
print("🏷️ Calculando flag_etapa_aprov...")

# Set normalizado (sem acento, maiúsculo) — adicionadas variantes reais do sistema
ETAPAS_APROV_NORM = {normalizar(e) for e in {
    # Deliberação SEONT
    "DELIBERAÇÃO SEONT",
    "DELIBERAÇÃO SEONT Z1",

    # Conferência final SEONT (com e sem acento — sistema usa sem)
    "SEONT CONFERÊNCIA FINAL - ANÁLISE TÉCNICA",
    "SEONT CONFERENCIA FINAL - ANALISE TECNICA",
    "SEONT CONFERÊNCIA FINAL - CONFERÊNCIA DOS DADOS",
    "SEONT CONFERENCIA FINAL - CONFERENCIA DOS DADOS",

    # Análise técnica SEONT (variantes encontradas nos dados)
    "SEONT - ANÁLISE TÉCNICA - CONFERÊNCIA DOS DADOS",
    "SEONT - ANALISE TECNICA - CONFERENCIA DOS DADOS",

    # Publicação oficial (etapa terminal)
    "REGISTRO DA PUBLICIDADE DO DIARIO OFICIAL",
    "REGISTRO DA PUBLICIDADE DO DIÁRIO OFICIAL",

    # Deferimento / Indeferimento
    "DEFERIMENTO",
    "INDEFERIMENTO",

    # Emissão
    "EMISSÃO ALVARÁ",
    "EMISSAO ALVARA",
    "EMISSÃO DA LICENÇA",
    "EMISSAO DA LICENCA",
    "EMISSÃO DA LICENÇA HABITE-SE",
    "EMISSAO DA LICENCA HABITE-SE",
    "EMISSÃO DE LICENÇA - EMPLAC",
    "EMISSAO DE LICENCA - EMPLAC",
    "EMISSÃO CARTA HABITAÇÃO PLURI",
    "EMISSAO CARTA HABITACAO PLURI",
    "ALVARÁ/LICENÇA DOC",
    "ALVARA/LICENCA DOC",

    # Finalização
    "FIM DE FLUXO",
    "FINALIZAR FLUXO",
    "FINALIZAÇÃO DO FLUXO",
    "FINALIZACAO DO FLUXO",
    "ETAPA RESULTADO FINAL",
    "ETAPA RESULTADO CONJUNTO FINAL",
}}

df["flag_etapa_aprov"] = (
    df["etapa_atual"]
    .apply(normalizar)
    .isin(ETAPAS_APROV_NORM)
    .astype(int)
)

print(f"   flag_etapa_aprov=1: {df['flag_etapa_aprov'].sum():,} registros")
print(f"   flag_etapa_aprov=0: {(df['flag_etapa_aprov'] == 0).sum():,} registros")
print()

# Diagnóstico: etapas SEONT que ficaram com flag=0
etapas_nao_mapeadas = (
    df.loc[(df["flag_seont"] == 1) & (df["flag_etapa_aprov"] == 0), "etapa_atual"]
    .value_counts()
    .head(20)
)
print("ℹ️ Etapas SEONT com flag_etapa_aprov=0 (verifique se alguma deve ser adicionada):")
print(etapas_nao_mapeadas.to_string())

StatementMeta(, d56499b8-ffb4-4190-8ea8-2cacac44b8d6, 8, Finished, Available, Finished, False)

🏷️ Calculando flag_etapa_aprov...
   flag_etapa_aprov=1: 2,630 registros
   flag_etapa_aprov=0: 8,296 registros

ℹ️ Etapas SEONT com flag_etapa_aprov=0 (verifique se alguma deve ser adicionada):
etapa_atual
SEONT CHEFIA - DISTRIBUIÇÃO       10
SEONT - PRÉ  ANÁLISE TECNICA       8
SEONT Z3 CHEFIA - DISTRIBUIÇÃO     2
SEONT Z2 CHEFIA - DISTRIBUIÇÃO     1
AGUARDANDO OUTROS                  1
SEONT Z1 CHEFIA - DISTRIBUIÇÃO     1
SEONT - PRÉ ANÁLISE TECNICA        1


In [7]:
# ============================================================
# 4c. executor_responsavel
# Réplica da fórmula Aux_Analista2 (coluna AN do Excel)
# Se etapa é SEONT e há executor → executor_atual
# Caso contrário → analista_responsavel (campo "Esta solicitação deverá ser analisada por:")
# ============================================================
print("👤 Calculando executor_responsavel...")

mask_seont = df["flag_seont"] == 1
mask_executor = (
    df["executor_atual"].notna()
    & (df["executor_atual"].astype(str).str.strip() != "")
    & (df["executor_atual"].astype(str).str.strip().str.upper() != "NAN")
)

df["executor_responsavel"] = df["executor_atual"].where(
    mask_seont & mask_executor,
    other=df.get("analista_responsavel")
)

preenchidos = df["executor_responsavel"].notna().sum()
nulos = df["executor_responsavel"].isna().sum()
print(f"   executor_responsavel preenchido: {preenchidos:,} ({preenchidos/len(df)*100:.1f}%)")
print(f"   executor_responsavel nulo:       {nulos:,} ({nulos/len(df)*100:.1f}%)")
print("\n✅ Colunas derivadas calculadas!")

StatementMeta(, d56499b8-ffb4-4190-8ea8-2cacac44b8d6, 9, Finished, Available, Finished, False)

👤 Calculando executor_responsavel...
   executor_responsavel preenchido: 476 (4.4%)
   executor_responsavel nulo:       10,450 (95.6%)

✅ Colunas derivadas calculadas!


In [8]:
print(f"📊 Total antes do filtro SEONT: {len(df):,} registros")

df_seont = df[df["flag_seont"] == 1].copy()

print(f"📊 Total após filtro SEONT:     {len(df_seont):,} registros")
print()

COLUNAS_FINAIS = [
    "n_da_solicitacao",
    "servico",
    "status",
    "data_criacao",
    "data_finalizacao",
    "solicitante",
    "titulo_profissional",
    "bairro_consolidado",
    "zona",
    "etapa_atual",
    "executor_atual",
    "analista_responsavel",     # substitui analista_z1/z2/z3
    "executor_responsavel",
    "aux_setor_responsavel",
    "data_etapa_inicio",
    "data_etapa_fim",
    "tempo_execucao",
    "dias_na_etapa",
    "flag_seont",
    "flag_etapa_aprov",
]

colunas_presentes = [c for c in COLUNAS_FINAIS if c in df_seont.columns]
colunas_descartadas = [c for c in df_seont.columns if c not in COLUNAS_FINAIS]

if colunas_descartadas:
    print(f"🗑️  Colunas descartadas: {colunas_descartadas}")

df_seont = df_seont[colunas_presentes]

print(f"\n✅ Schema final ({len(df_seont.columns)} colunas):")
print(df_seont.dtypes.to_string())

StatementMeta(, d56499b8-ffb4-4190-8ea8-2cacac44b8d6, 10, Finished, Available, Finished, False)

📊 Total antes do filtro SEONT: 10,926 registros
📊 Total após filtro SEONT:     196 registros

🗑️  Colunas descartadas: ['Executor', 'Etapa']

✅ Schema final (20 colunas):
n_da_solicitacao                  Int64
servico                          object
status                           object
data_criacao             datetime64[us]
data_finalizacao         datetime64[us]
solicitante                      object
titulo_profissional              object
bairro_consolidado               object
zona                             object
etapa_atual                      object
executor_atual                   object
analista_responsavel             object
executor_responsavel             object
aux_setor_responsavel            object
data_etapa_inicio        datetime64[us]
data_etapa_fim           datetime64[us]
tempo_execucao                   object
dias_na_etapa                   float64
flag_seont                        int64
flag_etapa_aprov                  int64


## 5. Filtro: apenas registros SEONT

In [9]:
print(f"📊 Total antes do filtro SEONT: {len(df):,} registros")

df_seont = df[df["flag_seont"] == 1].copy()

print(f"📊 Total após filtro SEONT:     {len(df_seont):,} registros")
print()

COLUNAS_FINAIS = [
    "n_da_solicitacao",
    "servico",
    "status",
    "data_criacao",
    "data_finalizacao",
    "solicitante",
    "titulo_profissional",
    "bairro_consolidado",
    "zona",
    "etapa_atual",
    "executor_atual",
    "analista_responsavel",     # substitui analista_z1/z2/z3
    "executor_responsavel",
    "aux_setor_responsavel",
    "data_etapa_inicio",
    "data_etapa_fim",
    "tempo_execucao",
    "dias_na_etapa",
    "flag_seont",
    "flag_etapa_aprov",
]

colunas_presentes = [c for c in COLUNAS_FINAIS if c in df_seont.columns]
colunas_descartadas = [c for c in df_seont.columns if c not in COLUNAS_FINAIS]

if colunas_descartadas:
    print(f"🗑️  Colunas descartadas: {colunas_descartadas}")

df_seont = df_seont[colunas_presentes]

print(f"\n✅ Schema final ({len(df_seont.columns)} colunas):")
print(df_seont.dtypes.to_string())

StatementMeta(, d56499b8-ffb4-4190-8ea8-2cacac44b8d6, 11, Finished, Available, Finished, False)

📊 Total antes do filtro SEONT: 10,926 registros
📊 Total após filtro SEONT:     196 registros

🗑️  Colunas descartadas: ['Executor', 'Etapa']

✅ Schema final (20 colunas):
n_da_solicitacao                  Int64
servico                          object
status                           object
data_criacao             datetime64[us]
data_finalizacao         datetime64[us]
solicitante                      object
titulo_profissional              object
bairro_consolidado               object
zona                             object
etapa_atual                      object
executor_atual                   object
analista_responsavel             object
executor_responsavel             object
aux_setor_responsavel            object
data_etapa_inicio        datetime64[us]
data_etapa_fim           datetime64[us]
tempo_execucao                   object
dias_na_etapa                   float64
flag_seont                        int64
flag_etapa_aprov                  int64


## 6. Validação antes de salvar

In [10]:
print("VALIDACAO gold_pdr_seont_os")
print("=" * 60)
print("Total de registros: " + str(len(df_seont)))
print("Total de colunas:   " + str(len(df_seont.columns)))
print()

assert df_seont["flag_seont"].eq(1).all(), "ERRO: Existem registros com flag_seont != 1!"
print("OK - Todos os registros sao SEONT (flag_seont=1)")

dist_aprov = df_seont["flag_etapa_aprov"].value_counts()
print("\nDistribuicao flag_etapa_aprov:")
for val, cnt in dist_aprov.items():
    pct = round(cnt / len(df_seont) * 100, 1)
    print("   " + str(val) + ": " + str(cnt) + " (" + str(pct) + "%)")

nulos_exec = df_seont["executor_responsavel"].isna().sum()
pct_nulos = round(nulos_exec / len(df_seont) * 100, 1)
print("\nNulos em executor_responsavel: " + str(nulos_exec) + " (" + str(pct_nulos) + "%)")

print("\nDistribuicao por aux_setor_responsavel:")
print(df_seont["aux_setor_responsavel"].value_counts().to_string())

print("\nAmostra dos dados:")
display(df_seont.head(5))

StatementMeta(, d56499b8-ffb4-4190-8ea8-2cacac44b8d6, 12, Finished, Available, Finished, False)

VALIDACAO gold_pdr_seont_os
Total de registros: 196
Total de colunas:   20

OK - Todos os registros sao SEONT (flag_seont=1)

Distribuicao flag_etapa_aprov:
   1: 172 (87.8%)
   0: 24 (12.2%)

Nulos em executor_responsavel: 54 (27.6%)

Distribuicao por aux_setor_responsavel:
aux_setor_responsavel
SEONT                 81
SEONT-Chefia (D.O)    69
SEONT-Chefia          46

Amostra dos dados:


SynapseWidget(Synapse.DataFrame, 0c264b1f-e12e-4f66-be65-229a946963b1)

In [11]:
df_seont.columns.tolist()

StatementMeta(, d56499b8-ffb4-4190-8ea8-2cacac44b8d6, 13, Finished, Available, Finished, False)

['n_da_solicitacao',
 'servico',
 'status',
 'data_criacao',
 'data_finalizacao',
 'solicitante',
 'titulo_profissional',
 'bairro_consolidado',
 'zona',
 'etapa_atual',
 'executor_atual',
 'analista_responsavel',
 'executor_responsavel',
 'aux_setor_responsavel',
 'data_etapa_inicio',
 'data_etapa_fim',
 'tempo_execucao',
 'dias_na_etapa',
 'flag_seont',
 'flag_etapa_aprov']

## 7. Salvar tabela gold_pdr_seont_os

In [ ]:
print("Convertendo para Spark e salvando gold_obras_seont_os...")

df_spark = spark.createDataFrame(df_seont)

df_spark.printSchema()

(
    df_spark.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("gold_obras_seont_os")
)

print("Tabela gold_obras_seont_os salva com sucesso!")
print("Registros: " + str(len(df_seont)) + " | Colunas: " + str(df_seont.columns.tolist()))

StatementMeta(, d56499b8-ffb4-4190-8ea8-2cacac44b8d6, 14, Finished, Available, Finished, False)

Convertendo para Spark e salvando gold_pdr_seont_os...
root
 |-- n_da_solicitacao: long (nullable = true)
 |-- servico: string (nullable = true)
 |-- status: string (nullable = true)
 |-- data_criacao: timestamp (nullable = true)
 |-- data_finalizacao: timestamp (nullable = true)
 |-- solicitante: string (nullable = true)
 |-- titulo_profissional: string (nullable = true)
 |-- bairro_consolidado: string (nullable = true)
 |-- zona: string (nullable = true)
 |-- etapa_atual: string (nullable = true)
 |-- executor_atual: string (nullable = true)
 |-- analista_responsavel: string (nullable = true)
 |-- executor_responsavel: string (nullable = true)
 |-- aux_setor_responsavel: string (nullable = true)
 |-- data_etapa_inicio: timestamp (nullable = true)
 |-- data_etapa_fim: timestamp (nullable = true)
 |-- tempo_execucao: string (nullable = true)
 |-- dias_na_etapa: double (nullable = true)
 |-- flag_seont: long (nullable = true)
 |-- flag_etapa_aprov: long (nullable = true)

Tabela gold_pd

In [ ]:
%%sql
SELECT
    COUNT(*)                                         AS total_registros,
    COUNT(DISTINCT n_da_solicitacao)                 AS os_distintas,
    SUM(flag_etapa_aprov)                            AS em_etapa_aprov,
    COUNT(CASE WHEN executor_responsavel IS NULL
               THEN 1 END)                           AS sem_executor_responsavel
FROM gold_obras_seont_os

StatementMeta(, d56499b8-ffb4-4190-8ea8-2cacac44b8d6, 15, Finished, Available, Finished, False)

<Spark SQL result set with 1 rows and 4 fields>